# 12 · Персистентность кластеров Wishart при `k=2 → 3 → 4` в фиксированном GloVe-пространстве

Анализирует уже рассчитанный sweep `wishart-glove-graphex-cn100k-k2-5-sweep-01` без повторной кластеризации. Проверяет, является ли `k=3` устойчивым мезоскопическим уровнем, или это перестройка density-estimator.

Метрики: переходы cluster↔noise, Jaccard/retention, split/merge entropy, ARI/NMI, цепочки `k2→k3→k4`, GloVe-компактность и дрейф центроидов. Это **cluster persistence**, не persistent homology.

In [ ]:
BRANCH="feature/glove-wishart-graphex-cn100k-lossless-l4"
DRIVE_ROOT="/content/drive/MyDrive/SemanticMap/colab/wishart"
SWEEP_RUN="wishart-glove-graphex-cn100k-k2-5-sweep-01"
K_VALUES=(2,3,4)
EXPECTED_NODES=100000
SCRATCH_ROOT="/content/semmap-glove-persistence-k2-k4"
MIN_LINEAGE_SUPPORT=20
MIN_PAIR_JACCARD=0.25

In [ ]:
import os,sys,shutil,subprocess,json,math
from pathlib import Path
ROOT=Path(SCRATCH_ROOT); ROOT.mkdir(parents=True,exist_ok=True)
REPO=ROOT/'repo'
if not (REPO/'.git').exists(): subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/SemanticMap/semgraphex.git',str(REPO)],check=True)
COMMIT=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','scipy','scikit-learn','matplotlib'],check=True)
print('commit',COMMIT)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd
DRIVE=Path(DRIVE_ROOT); REMOTE=DRIVE/'runs'/SWEEP_RUN; LOCAL=ROOT/'input'; OUT=ROOT/'output'; OUT_REMOTE=REMOTE/'persistence_k2_k3_k4'
if not (REMOTE/'COMPLETED').is_file(): raise RuntimeError('sweep run not COMPLETED')
LOCAL.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
for src,name in [(REMOTE/'k_02'/'embedding'/'vectors.npy','vectors.npy'),(REMOTE/'k_02'/'embedding'/'coverage.npy','coverage.npy')]:
    dst=LOCAL/name
    if not dst.is_file() or dst.stat().st_size!=src.stat().st_size: shutil.copy2(src,dst)
for k in K_VALUES:
    src=REMOTE/f'k_{k:02d}'/'knn'/'labels.npy'; dst=LOCAL/f'labels_k{k}.npy'
    if not dst.is_file() or dst.stat().st_size!=src.stat().st_size: shutil.copy2(src,dst)
vectors=np.load(LOCAL/'vectors.npy',mmap_mode='r'); coverage=np.load(LOCAL/'coverage.npy'); labels={k:np.load(LOCAL/f'labels_k{k}.npy') for k in K_VALUES}
assert vectors.shape[0]==EXPECTED_NODES
for k in K_VALUES: assert labels[k].shape==(EXPECTED_NODES,)
print(vectors.shape,{k:(len(np.unique(labels[k][labels[k]>=0])),int((labels[k]<0).sum())) for k in K_VALUES})

In [ ]:
from collections import defaultdict
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score
def sizes(x):
    u,c=np.unique(x[x>=0],return_counts=True); return dict(zip(u.astype(int),c.astype(int)))
def H(c):
    c=np.asarray(c,float); p=c[c>0]/c.sum() if c.sum() else np.array([]); return float(-(p*np.log2(p)).sum()) if len(p) else 0.0
def transitions(a,b,ka,kb):
    sa,sb=sizes(a),sizes(b); pairs=np.stack([a,b],1); u,c=np.unique(pairs,axis=0,return_counts=True); ct=pd.DataFrame({'source':u[:,0].astype(int),'target':u[:,1].astype(int),'intersection':c.astype(int)})
    rows=[]
    for s,g in ct[ct.source>=0].groupby('source'):
        s=int(s); cand=g[g.target>=0].copy(); best_t=-1; bj=br=bi=0
        if len(cand):
            cand['ts']=cand.target.map(sb); cand['j']=cand.intersection/(sa[s]+cand.ts-cand.intersection); z=cand.sort_values(['j','intersection'],ascending=False).iloc[0]; best_t=int(z.target); bj=float(z.j); bi=int(z.intersection); br=bi/sa[s]
        dist=g.intersection.to_numpy(); mh=math.log2(len(dist)) if len(dist)>1 else 0
        rows.append({'k_from':ka,'k_to':kb,'source_cluster':s,'source_size':sa[s],'best_target':best_t,'best_intersection':bi,'best_jaccard':bj,'best_retention':br,'to_noise_fraction':int(g.loc[g.target<0,'intersection'].sum())/sa[s],'split_entropy_norm':H(dist)/mh if mh else 0})
    summary={'k_from':ka,'k_to':kb,'ARI_all':float(adjusted_rand_score(a,b)),'NMI_all':float(normalized_mutual_info_score(a,b)),'clustered_to_noise':int(((a>=0)&(b<0)).sum()),'noise_to_clustered':int(((a<0)&(b>=0)).sum()),'both_clustered':int(((a>=0)&(b>=0)).sum())}
    return ct,pd.DataFrame(rows),summary
pairs={}; summaries=[]
for ka,kb in [(2,3),(3,4)]:
    ct,src,s=transitions(labels[ka],labels[kb],ka,kb); pairs[(ka,kb)]={'ct':ct,'src':src}; summaries.append(s)
display(pd.DataFrame(summaries))

In [ ]:
def geometry(vectors,lab):
    valid=lab>=0; ids=np.unique(lab[valid]).astype(int); m=int(ids.max()) if len(ids) else -1
    sums=np.zeros((m+1,vectors.shape[1]),np.float64); cnt=np.bincount(lab[valid],minlength=m+1)
    idx=np.flatnonzero(valid)
    for s in range(0,len(idx),20000):
        take=idx[s:s+20000]; np.add.at(sums,lab[take],np.asarray(vectors[take],dtype=np.float64))
    norms=np.linalg.norm(sums,axis=1); cent=np.zeros_like(sums); nz=norms>0; cent[nz]=sums[nz]/norms[nz,None]; comp=np.zeros_like(norms); comp[nz]=norms[nz]/cnt[nz]
    return pd.DataFrame({'cluster':ids,'size':cnt[ids],'glove_compactness':comp[ids],'mean_cosine_distance_to_centroid':1-comp[ids]}),cent
geom={}; cents={}
for k in K_VALUES: geom[k],cents[k]=geometry(vectors,labels[k]); print(k,'median compactness',geom[k].glove_compactness.median())

In [ ]:
m23=pairs[(2,3)]['src'].set_index('source_cluster'); m34=pairs[(3,4)]['src'].set_index('source_cluster'); s2,s3,s4=sizes(labels[2]),sizes(labels[3]),sizes(labels[4]); rows=[]
for c2,r23 in m23.iterrows():
    c3=int(r23.best_target)
    if c3<0 or c3 not in m34.index: continue
    r34=m34.loc[c3]; c4=int(r34.best_target)
    if c4<0: continue
    triple=int(((labels[2]==int(c2))&(labels[3]==c3)&(labels[4]==c4)).sum())
    if not triple: continue
    d23=float(1-np.clip(cents[2][int(c2)]@cents[3][c3],-1,1)); d34=float(1-np.clip(cents[3][c3]@cents[4][c4],-1,1)); d24=float(1-np.clip(cents[2][int(c2)]@cents[4][c4],-1,1))
    rows.append({'cluster_k2':int(c2),'cluster_k3':c3,'cluster_k4':c4,'size_k2':s2[int(c2)],'size_k3':s3[c3],'size_k4':s4[c4],'jaccard_2_3':float(r23.best_jaccard),'jaccard_3_4':float(r34.best_jaccard),'triple_intersection':triple,'triple_fraction_of_k2':triple/s2[int(c2)],'centroid_cosine_distance_2_3':d23,'centroid_cosine_distance_3_4':d34,'centroid_cosine_distance_2_4':d24,'persistent_by_threshold':triple>=MIN_LINEAGE_SUPPORT and float(r23.best_jaccard)>=MIN_PAIR_JACCARD and float(r34.best_jaccard)>=MIN_PAIR_JACCARD})
lineages=pd.DataFrame(rows); print('lineages',len(lineages),'persistent',int(lineages.persistent_by_threshold.sum()) if len(lineages) else 0); display(lineages.sort_values(['persistent_by_threshold','triple_intersection'],ascending=False).head(50))

In [ ]:
import matplotlib.pyplot as plt
summary=[]
for k in K_VALUES: summary.append({'k':k,'clusters':len(np.unique(labels[k][labels[k]>=0])),'noise_fraction':float((labels[k]<0).mean())})
sk=pd.DataFrame(summary)
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(sk.k,sk.clusters,marker='o'); ax.set_xlabel('k_neighbors'); ax.set_ylabel('clusters'); ax.set_title('Wishart cluster count vs k'); ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(OUT/'cluster_count_vs_k.png',dpi=160); plt.show()
fig,ax=plt.subplots(figsize=(8,5))
for ka,kb in [(2,3),(3,4)]:
    f=pairs[(ka,kb)]['src']; ax.scatter(f.source_size,f.best_jaccard,s=10,alpha=.35,label=f'{ka}→{kb}')
ax.set_xscale('log'); ax.set_xlabel('source cluster size'); ax.set_ylabel('best Jaccard'); ax.set_title('Cluster persistence across k'); ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); fig.savefig(OUT/'cluster_jaccard_vs_size.png',dpi=160); plt.show()
if len(lineages):
    fig,ax=plt.subplots(figsize=(8,5)); ax.scatter(lineages.triple_fraction_of_k2,lineages.centroid_cosine_distance_2_4,s=np.clip(np.sqrt(lineages.triple_intersection)*3,8,120),alpha=.45); ax.set_xlabel('triple intersection / size(k=2)'); ax.set_ylabel('GloVe centroid cosine distance k2→k4'); ax.set_title('Persistence vs geometric drift'); ax.grid(alpha=.2); fig.tight_layout(); fig.savefig(OUT/'lineage_persistence_vs_centroid_drift.png',dpi=160); plt.show()

In [ ]:
# Save all artifacts and publish COMPLETED last
for (ka,kb),o in pairs.items(): o['ct'].to_csv(OUT/f'contingency_k{ka}_k{kb}.csv',index=False); o['src'].to_csv(OUT/f'source_persistence_k{ka}_k{kb}.csv',index=False)
pd.DataFrame(summaries).to_csv(OUT/'pair_summary.csv',index=False); pd.concat([g.assign(k=k) for k,g in geom.items()]).to_csv(OUT/'cluster_glove_geometry.csv',index=False); lineages.to_csv(OUT/'lineages_k2_k3_k4.csv',index=False); sk.to_csv(OUT/'k_summary.csv',index=False)
report={'experiment':SWEEP_RUN,'code_revision':COMMIT,'k_values':list(K_VALUES),'lineages_total':int(len(lineages)),'persistent_lineages_threshold':int(lineages.persistent_by_threshold.sum()) if len(lineages) else 0,'thresholds':{'min_lineage_support':MIN_LINEAGE_SUPPORT,'min_pair_jaccard':MIN_PAIR_JACCARD},'pairwise':summaries,'contract':{'fixed_glove_embedding':True,'no_reclustering':True,'cluster_persistence_not_persistent_homology':True}}
(OUT/'persistence_summary.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf8')
OUT_REMOTE.mkdir(parents=True,exist_ok=True)
for x in OUT.iterdir():
    if x.is_file():
        tmp=OUT_REMOTE/(x.name+'.part'); shutil.copy2(x,tmp); os.replace(tmp,OUT_REMOTE/x.name)
(OUT_REMOTE/'COMPLETED').write_text('validated\n',encoding='ascii')
print('ГОТОВО',OUT_REMOTE); print(json.dumps(report,ensure_ascii=False,indent=2))

## Критерий интерпретации

Гипотеза о мезоскопическом уровне `k=3` получает поддержку, если одновременно видны: высокий Jaccard `2→3` и `3→4`, большие тройные пересечения, низкий split entropy и малый дрейф GloVe-центроидов. Если `k=3` в основном состоит из массовых merges/splits с большим centroid drift, это скорее чувствительность оценки плотности к масштабу.